In [1]:
%%bash
echo "🔧 Setting up Python 3.10 environment for Stage 3 (XTTS)..."
apt-get update -q
apt-get install -y python3.10 python3.10-distutils -q
curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10

echo "📦 Installing XTTS and Server Dependencies inside Python 3.10..."
python3.10 -m pip install -q --ignore-installed blinker TTS==0.22.0 torchcodec "transformers==4.44.2"
python3.10 -m pip install -q fastapi uvicorn python-multipart pyngrok pydantic

echo "✅ Environment Setup Complete!"

🔧 Setting up Python 3.10 environment for Stage 3 (XTTS)...
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,812 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%writefile server_stage3_tts.py
import os
import requests
import torch
from pydantic import BaseModel
from fastapi import FastAPI
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from TTS.api import TTS
import uvicorn
from pyngrok import ngrok

_original_load = torch.load
def _patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_load(*args, **kwargs)
torch.load = _patched_load

PROJECT_DIR = '/content/drive/MyDrive/Video_Translation_Project'
MODELS_CACHE = os.path.join(PROJECT_DIR, 'Models_Cache')
os.environ['HF_HOME'] = MODELS_CACHE
os.environ['XDG_DATA_HOME'] = MODELS_CACHE
os.environ['COQUI_TOS_AGREED'] = "1"

device = "cuda" if torch.cuda.is_available() else "cpu"
app = FastAPI(title="Stage 3: TTS API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

print("[INFO] Loading XTTS (Voice Cloning)...")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("[INFO] XTTS loaded successfully!")

class TTSRequest(BaseModel):
    arabic_text: str
    vocals_url: str

@app.post("/process_stage3_tts")
async def process_stage3_tts(payload: TTSRequest):
    input_audio = "/content/local_vocals.wav"
    final_synced_output = "/content/final_arabic_synced.wav"

    print("[INFO] Downloading clean vocals for voice cloning reference...")
    res = requests.get(payload.vocals_url, headers={'ngrok-skip-browser-warning': 'true'})
    with open(input_audio, 'wb') as f:
        f.write(res.content)

    print("[INFO] Synthesizing audio...")
    tts.tts_to_file(
        text=payload.arabic_text,
        file_path=final_synced_output,
        speaker_wav=input_audio,
        language="ar",
        split_sentences=True
    )

    print("[INFO] Stage 3 Complete!")
    return {"message": "Audio synthesized successfully."}

@app.get("/get_arabic_audio")
def get_arabic_audio():
    return FileResponse("/content/final_arabic_synced.wav", media_type="audio/wav")

if __name__ == "__main__":
    ngrok.set_auth_token("")
    public_url = ngrok.connect(8004).public_url
    print(f"\n🚀 STAGE 3 (XTTS) MICROSERVICE IS LIVE AT: {public_url}\n")
    uvicorn.run(app, host="0.0.0.0", port=8004)

Writing server_stage3_tts.py


In [ ]:
!python3.10 server_stage3_tts.py

[INFO] Loading XTTS (Voice Cloning)...
 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts
[INFO] XTTS loaded successfully!

🚀 STAGE 3 (XTTS) MICROSERVICE IS LIVE AT: https://generic-audition-chaos.ngrok-free.dev

INFO:     Started server process [30856]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8004 (Press CTRL+C to quit)
INFO:     197.63.240.96:0 - "OPTIONS /process_stage3_tts HTTP/1.1" 200 OK
[INFO] Downloading clean vocals for voice cloning reference...
[INFO] Synthesizing audio...
 > Text splitted to sentences.
['مرحباً dear friends!', 'لقد مر بضع سنوات، هل ترغب في مشاركة ما تمكنا من تحقيقه؟ اليوم سنتحدث عن رحلتنا.', 'فما الذي نقوم به يا أصدقائي؟ سنستكشف جميع الأعالي.', 'ولكن أولًا ، سنبدأ من آسيا وأوروبا و.']
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected 